###**Coleta dos Dados Bioclimáticos (2012-2024)**

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Aponta para a RAIZ do projeto
%cd /content/drive/MyDrive/cafeicultura-varginha-analytics

In [ ]:
# coleta de dados bioclimáticos (2012-2024)  do INMET (Instituto Nacional de Meteorologia)

import os
import requests

# 1. Definindo intervalo de anos do projeto
anos = range(2012, 2025)
base_url = "https://portal.inmet.gov.br/uploads/dadoshistoricos"
pasta_raw = "data/raw"

print("Iniciando o download dos dados históricos do INMET (2012-2024)...\n")

# 2. Loop para baixar cada pacote anual
for ano in anos:
    url = f"{base_url}/{ano}.zip"
    caminho_zip = os.path.join(pasta_raw, f"{ano}.zip")

    if not os.path.exists(caminho_zip):
        print(f"Baixando {ano}.zip...")
        resposta = requests.get(url, stream=True)

        if resposta.status_code == 200:
            with open(caminho_zip, "wb") as f:
                for chunk in resposta.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"✔ {ano}.zip salvo com sucesso.")
        else:
            print(f"✖ Falha ao baixar {ano}.zip (Status Code: {resposta.status_code})")
    else:
        print(f"➜ Arquivo {ano}.zip já existe em {pasta_raw}.")

print("\nDownload concluído!")

In [ ]:
# Filtrando e extraindo apenas a estação de Varginha-MG

import zipfile

estacao_varginha = "VARGINHA"
pasta_extraida = "data/raw/varginha_raw"
os.makedirs(pasta_extraida, exist_ok=True)

for ano in anos:
    caminho_zip = os.path.join(pasta_raw, f"{ano}.zip")

    if os.path.exists(caminho_zip):
        with zipfile.ZipFile(caminho_zip, "r") as zip_ref:
            # Procura no zip pelo arquivo CSV que contém VARGINHA no nome
            arquivos = zip_ref.namelist()
            arquivo_varginha = [
                f for f in arquivos if estacao_varginha in f.upper()
            ]

            if arquivo_varginha:
                # Extrai apenas o CSV de Varginha
                zip_ref.extract(arquivo_varginha[0], pasta_extraida)
                print(f"✔ Dados de Varginha ({ano}) extraídos com sucesso.")
            else:
                print(f"⚠️ Estação de Varginha não encontrada no pacote de {ano}.")

In [ ]:
# Unificando os arquivos anuais em um único DataFrame

import glob
import pandas as pd

# 1. Localiza todos os arquivos CSV extraídos da pasta de Varginha
arquivos_csv = glob.glob(
    "data/raw/varginha_raw/**/*.CSV", recursive=True
) + glob.glob("data/raw/varginha_raw/**/*.csv", recursive=True)

lista_dfs = []

# 2. Lê cada arquivo aplicando os ajustes necessários do INMET
for arquivo in sorted(arquivos_csv):
    df = pd.read_csv(
        arquivo, sep=";", skiprows=8, encoding="latin-1", decimal=","
    )
    lista_dfs.append(df)

# 3. Une todos os anos em um único DataFrame
df_varginha = pd.concat(lista_dfs, ignore_index=True)

print(
    f"✔ Dados consolidados! Total de linhas carregadas: {len(df_varginha):,}"
)
df_varginha.head(3)

###**Unificar os arquivos anuais em um único DataFrame**

In [ ]:
import glob
import pandas as pd

# 1. Localiza todos os arquivos CSV extraídos da pasta de Varginha
arquivos_csv = glob.glob(
    "data/raw/varginha_raw/**/*.CSV", recursive=True
) + glob.glob("data/raw/varginha_raw/**/*.csv", recursive=True)

lista_dfs = []

# 2. Lê cada arquivo aplicando os ajustes necessários do INMET
for arquivo in sorted(arquivos_csv):
    df = pd.read_csv(
        arquivo, sep=";", skiprows=8, encoding="latin-1", decimal=","
    )
    lista_dfs.append(df)

# 3. Une todos os anos em um único DataFrame
df_varginha = pd.concat(lista_dfs, ignore_index=True)

print(
    f"✔ Dados consolidados! Total de linhas carregadas: {len(df_varginha):,}"
)
df_varginha.head(3)

###**Padronização de colunas, formatação de Data/Hora e salvamento do dataset unificado.**
1. Limpeza de colunas e criação da variável temporal


In [ ]:
# 1. Mapeamento de renomeação
colunas_renomear = {
    'Data': 'data',
    'DATA (YYYY-MM-DD)': 'data',
    'Hora UTC': 'hora',
    'HORA (UTC)': 'hora',
    'PRECIPITAÇÃO TOTAL, HORÁRIA (mm)': 'precipitacao_mm',
    'PRECIPITACAO TOTAL, HORARIA (mm)': 'precipitacao_mm',
    'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)': 'temp_ar_c',
    'TEMPERATURA DO AR - BULBO SECO, HORARIA (Â°C)': 'temp_ar_c',
    'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)': 'temp_max_c',
    'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)': 'temp_min_c',
    'UMIDADE RELATIVA DO AR, HORARIA (%)': 'umidade_rel_pct',
    'VENTO, VELOCIDADE (m/s)': 'vento_velocidade_ms',
}

df_varginha = df_varginha.rename(columns=colunas_renomear)

# 2. Consolida colunas duplicadas criadas pelo rename
df_varginha = df_varginha.T.groupby(level=0).first().T

# 3. Trata a coluna 'hora' e cria 'data_hora'
df_varginha['hora_limpa'] = (
    df_varginha['hora']
    .astype(str)
    .str.replace(' UTC', '')
    .str.zfill(4)
    .str[:2]
    + ':00'
)

df_varginha['data_hora'] = pd.to_datetime(
    df_varginha['data'].astype(str) + ' ' + df_varginha['hora_limpa'],
    format='mixed',
    errors='coerce',
)

# 4. Adiciona colunas temporais separadas
df_varginha['ano'] = df_varginha['data_hora'].dt.year
df_varginha['mes'] = df_varginha['data_hora'].dt.month
df_varginha['dia'] = df_varginha['data_hora'].dt.day
df_varginha['hora_num'] = df_varginha['data_hora'].dt.hour
df_varginha['dia_do_ano'] = df_varginha['data_hora'].dt.dayofyear

# 5. Ordena e salva o dataset bruto consolidado
df_varginha = df_varginha.sort_values('data_hora').reset_index(drop=True)
caminho_saida = 'data/raw/varginha_2012_2024_consolidado.csv'
df_varginha.to_csv(caminho_saida, index=False, encoding='utf-8')

print(f'✔ Dataset consolidado salvo em: {caminho_saida}')

2. Versionamento no Git


In [ ]:
!git add 01_coleta_dados.ipynb
!git commit -m "feat: adiciona coleta e consolidacao dos dados do INMET"
!git push origin main